In [ ]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_pareto

from plotly.subplots import make_subplots
from pygmo import fast_non_dominated_sorting

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pickle
import plotly.colors as pc
import plotly.graph_objects as go

# Microgrid 3D Plotting:

In [ ]:
position_dim = 3

# Lead_Acid(0) Li-ion(1) ZEBRA(2) NaS(3) NiCd(4) NiMH(5) RFV(6) ZnBr(7)
bat_name = ['Lead_Acid', 'Li-ion', 'ZEBRA', 'NaS', 'NiCd', 'NiMH', 'RFV', 'ZnBr']
plot_name = ['Lead Acid', 'Li-ion', 'ZEBRA', 'NaS', 'NiCd', 'NiMH', 'RFV', 'ZnBr']
select_bat = [0,1,2,3,4,5,6,7] # [0,1,2,3,4,5,6,7]

# Choosing the files
choose_global_best_attribution_type = [0,1] # [0, 1, 2 ,3]
choose_dm_pool_type = [0,1,2] # [0, 1, 2]
choose_de_mutation_type = [0,1,2,3,4] # [0, 1, 2, 3, 4]

# Name of chosen files
file_names = [
        [
        f"MESH_G{i+1}S{j+1}D{k+1}_{bat_name[b]}_3_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
      ] for b in select_bat
    ]

num_batteries = len(file_names) # Number of dataset

##### Color setup #####
solid_colors = [mcolors.to_hex(cm.tab10(i+1)) for i in range(num_batteries)] # List with solid colors
colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'diamond-open', 'square-open', 'circle-open']
num_symbols = len(marker_symbols)
#######################

# Collecting data from files
idxs = [0]
count = 0
battery_solutions = np.empty((0,position_dim))
battery_pareto = np.empty((0,3))
for i in range(num_batteries):
	for j in range(len(file_names[i])):
		# Merging all solutions and pareto fronts for the same battery type
		with open("../results/" + file_names[i][j], "rb") as f:
			r = pickle.load(f)
			battery_solutions = np.vstack((battery_solutions, r['combined'][0]))
			battery_pareto = np.vstack((battery_pareto, r['combined'][1]))
			count += len(r['combined'][0])
	idxs.append(count)
# Getting the non-dominated front indices
nds_indices = fast_non_dominated_sorting(battery_pareto)[0][0]

## Pareto Front

In [ ]:
pareto_front = []

# Labeling the solutions with the battery names
for i in range(num_batteries):
	nds_idxs = nds_indices[np.where((nds_indices >= idxs[i]) & (nds_indices < idxs[i+1]))[0]]
	nds_battery_solutions = battery_solutions[nds_idxs]
	nds_battery_pareto = battery_pareto[nds_idxs]
	# Creating the plots
	pareto_front.append(
	go.Scatter3d(x=nds_battery_pareto[:,0],
				y=-nds_battery_pareto[:,1],
				z=nds_battery_pareto[:,2],
				mode='markers', marker=dict(size=4, color=solid_colors[i], symbol=marker_symbols[i]),
				name=plot_name[select_bat[i]])
	)

fig = go.Figure()

for i in range(num_batteries):
    fig.add_trace(pareto_front[i])

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1.5,y=1.5,z=1.5)
        ),
        xaxis=dict(
            title=dict(text='LCOE [US$/kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='RF', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='MEEF', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14),
        itemsizing='constant'
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=800,   # Figure width (pixels)
    height=600,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

## Design Space

In [ ]:
design_space = []

# Labeling the solutions with the battery names
for i in range(num_batteries):
	nds_idxs = nds_indices[np.where((nds_indices >= idxs[i]) & (nds_indices < idxs[i+1]))[0]]
	nds_battery_solutions = battery_solutions[nds_idxs]
	# Creating the plots
	design_space.append(
	go.Scatter3d(x=nds_battery_solutions[:,0],
				y=nds_battery_solutions[:,1],
				z=nds_battery_solutions[:,2],
				mode='markers', marker=dict(size=4, color=solid_colors[i], symbol=marker_symbols[i]),
				name=plot_name[select_bat[i]])
	)

fig = go.Figure()

for i in range(num_batteries):
    fig.add_trace(design_space[i])

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='Potência nominal do P. F. [kWp]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Potência nominal das turbinas [kW]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Capacidade da bateria [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        )
    ),
    legend=dict(
        x=0.95,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14),
        itemsizing='constant'
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=800,   # Figure width (pixels)
    height=600,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

# Benchmark 2D:

In [ ]:
objective_dim = 2 # Number of objectives
position_dim = 10 # Number of variables
n_pareto_points = 5000 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "zdt6"

# Choosing the files
insert_old_mesh = True
insert_nsga2 = True

# Choosing the MESH files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_dm_pool_type = [2] # [0, 1, 2]
choose_de_mutation_type = [1] # [0, 1, 2, 3, 4]

# Chosing the Old MESH files
old_global_best_attribution_type = 0 # [0, 1, 2, 3]
old_dm_pool_type = 2 # [0, 1, 2]
old_de_mutation_type = 1 # [0, 1, 2, 3, 4]

file_tuple = []

# NSGA-II files
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA2'))
# MESH files
file_tuple.extend([
        (f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}')
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
    ])
# OLD MESH files
if insert_old_mesh:
	file_tuple.append((f'OLD_MESH_G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MESH - ' + f'G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1} (Orig.)'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
solid_colors = [mcolors.to_hex(cm.tab10(i)) for i in range(num_datasets)] # List with solid colors
colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   mode='markers', 
                   marker=dict(size=8, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [ ]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.05, 1.05]
# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    xaxis=dict(title=dict(text='f1', font=dict(size=14)), tickfont=dict(size=14)),
    yaxis=dict(title=dict(text='f2', font=dict(size=14)), tickfont=dict(size=14)),
    # title=dict(
    #     text='MESH - ' + experiment_name.upper(),
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=480,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_points, position_dim, 2, wfg_k)
fig.add_trace(
	go.Scatter(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()

# Benchmark 3D:

In [ ]:
objective_dim = 3 # Number of objectives
position_dim = 10 # Number of variables
n_pareto_density = 100 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "wfg9"

# Choosing the files
insert_old_mesh = True
insert_nsga2 = True

# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_dm_pool_type = [0] # [0, 1, 2]
choose_de_mutation_type = [0] # [0, 1, 2, 3, 4]

# Chosing the Old MESH files
old_global_best_attribution_type = 1 # [0, 1, 2, 3]
old_dm_pool_type = 0 # [0, 1, 2]
old_de_mutation_type = 3 # [0, 1, 2, 3, 4]

file_tuple = []

# NSGA-II files
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA2'))
# MESH files
file_tuple.extend([
        (f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}')
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
    ])
# OLD MESH files
if insert_old_mesh:
	file_tuple.append((f'OLD_MESH_G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl',
                    'MESH - ' + f'G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1} (Orig.)'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
solid_colors = [mcolors.to_hex(cm.tab10(i)) for i in range(num_datasets)] # List with solid colors
colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter3d(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   z=r['combined'][1][:,2],
                   mode='markers', marker=dict(size=5, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [ ]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.1, 1.2]

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1,y=1,z=1)
        ),
        xaxis=dict(
            title=dict(text='f1', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # X-axis background
            gridcolor='lightgray',    # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='f2', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Y-axis background
            gridcolor='lightgray',    # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='f3', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Z-axis background
            gridcolor='lightgray',    # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=420,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_density, position_dim, 3, wfg_k)
fig.add_trace(
	go.Scatter3d(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		z=real_pareto_front[:,2],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()